# Exploratory Data Analysis

## Contents

| Section | Description |
|---|---|
| [1. Problem and Data Overview](#1.-Problem-and-Data-Overview) | Define the prediction problem and inspect the dataset structure. |
| [2. Data Quality Checks](#2.-Data-Quality-Checks) | Check missing data, duplicates, invalid values, and feature types. |
| [3. Train/Test Split](#3.-Train/Test-Split) | Create an untouched holdout set before making modelling decisions. |
| [4. Target Analysis](#4.-Target-Analysis) | Understand the target distribution and any imbalance or skew. |
| [5. Univariate Analysis](#5.-Univariate-Analysis) | Examine individual numerical and categorical features. |
| [6. Feature/Target Relationships](#6.-Feature/Target-Relationships) | Identify features that appear related to the target. |
| [7. Feature/Feature Relationships](#7.-Feature/Feature-Relationships) | Check correlations, redundancy, and potential interactions. |
| [8. Outliers and Skew](#8.-Outliers-and-Skew) | Identify unusual observations and potentially useful transformations. |
| [9. Feature Engineering Ideas](#9.-Feature-Engineering-Ideas) | Record useful features suggested by the exploratory analysis. |
| [10. EDA Conclusions](#10.-EDA-Conclusions) | Translate findings into concrete preprocessing decisions. |

---

In [ ]:
# Basic libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn libraries
from sklearn.model_selection import train_test_split

---
## 1. Problem and Data Overview

| Subsection | Description |
|---|---|
| [1.1 Problem Definition](#1.1-Problem-Definition) | State the target and whether the task is regression or classification. |
| [1.2 Dataset Basic Info](#1.2-Dataset-Basic-Info) | Check the number of observations and features. |
| [1.3 Dataset Overview](#1.3-Dataset-Overview) | Inspect column names, meanings, and likely feature roles. |


### 1.1 Problem Definition
State the overall target for the analysis (i.e. the variable we want to predict or understand), and note whether this is a classification or a regression task.

Any useful bits of information you know prior to beginning the analysis can go here.

We can then **Load the Data:**

In [ ]:
import pandas as pd

df = pd.read_csv(
    'data.csv', # or url
    na_values = ['N/A', 'Unknown'],
    usecols = None,
    skiprows = None,
    index_col = None,
    sep = ',',
    parse_dates = None
)

- ```na_values``` replaces specified strings with NaN
- ```usecols``` retrieves only selected columns from file
- ```skiprows``` skips rows in dataframe (e.g. if first few lines are unnecessary)
- ```index_col``` uses specified column to index (if none assigns an integer)
- ```sep``` allows Pandas to handle multiple delimiters in file
- ```parse_dates``` converts date columns into datetime objects


### 1.2 Dataset Basic Info
**Shape** will tell you how many features and datapoints are in the dataset. Datatypes will be useful to record in the column description in the next section.

**Info** gives you details on potentially missing values, as well as each feature's datatype.

- Note any potentially missing values.
- Note datatype for column overview

In [ ]:
print("Shape:")
print(f"rows={df.shape[0]}, columns={df.shape[1]}")

print()
print("Info:")
df.info()

Basic statistical summary of features and target variable:

In [ ]:
# Basic description of numerical columns
df.agg(["count","min","max"])

# Statistical summary of string-valued columns
df.describe(include="str")

If any features have a handful of values, you can look at them here:

In [ ]:
# Categories and counts
df["categorical_column"].value_counts()

### 1.3 Dataset Overview

First, we look at the head (and potentially tail) of the distribution.

In [ ]:
# First n=10 rows
df.head(n=10)

# Last n=10 rows
df.tail(n=10)

Then we **create description table:**

|Column|Description|Values|Notes|
|-|-|-|-|
| Column name | Brief description | Datatype / values taken | Relevant notes |

Make sure to include any relevant details you have noticed previously, and include **obvservations**.

---
## 2. Data Quality Checks

| Subsection | Description |
|---|---|
| [2.1 Missing Values](#3.1-Missing-Values) | Measure missingness by feature. |
| [2.2 Duplicate Rows](#3.2-Duplicate-Rows) | Identify repeated observations. |
| [2.3 Invalid Values](#3.3-Invalid-Values) | Check impossible or suspicious values. |
| [2.4 Category Consistency](#3.4-Category-Consistency) | Look for duplicated labels, spelling differences, or formatting issues. |
| [2.5 Redundant Features](#3.5-Redundant-Features) | Identify columns that are unlikely to help prediction. |

### 2.1 Missing Values

**Missing value table:**

In [ ]:
# Create a new dataframe containing the sum of missing values, along with
# the percentage of missing values
missing_vals = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().sum() / len(df) * 100).round(2)
})
# Remove all rows without missing values, then sort according to the
# missing value percentage
missing_vals = missing_vals[missing_vals["Missing Count"] > 0].sort_values("Missing %", ascending=False)
# Print the final result
missing_vals

**Missing value histogram:**

In [ ]:
plt.subplots(figsize=(10, 4))
plt.barh(df.columns, df.isnull().sum())
for idx, val in enumerate(df.isnull().sum()):
    if val > 0:
        plt.text(val + 5, idx, str(val), va="center")
plt.title("# Missing Values")
plt.tight_layout()
plt.show()

**Make a note of what to drop and what to impute**

Drop row:
- Small fraction of rows missing
    - < 5% dropping probably fine
    - 5% - 10% investigate before dropping
    - \> 10% be careful
- Randomly distributed data (i.e. not concentrated on a single group)
- Not important enough to impute

Drop column:
- Large fraction of data is missing (>40% - 60%)
- Feature not important
- Unreliable reconstruction
- Redundant column

Numerical imputation:
- ```median``` is a safe default (especially for skewed data or containing outliers)
- ```mean``` when distribution is symmetric or has minimal outliers

Categorical imputation:
- ```most_frequent``` is a good default (i.e. mode)
- ```fillna("Missing")``` for when missingness contains information
- ```df["missing_indicator"]``` for when we want to track missing data, i.e. missingness may be useful for predicting target (good for tree-based models)

**NB:** Never calculate imputation values on the full dataset.

### 2.2 Duplicate Rows
Obviously need to drop duplicated rows, as they over-represent datapoints.

However, need to take care with this, as the relevant target variables need to be dropped as well.

In [ ]:
# Count the number of duplicated rows:
X_train.duplicated().sum()

# View the duplicated rows:
X_train[X_train.duplicated()]

# Drop duplicates:
df.drop_duplicates()

### 2.3 Invalid Values
Check for impossible or invalid values. This will have to be done on a case-by-case basis.

### 2.4 Category Consistency
Look for duplicated labels, spelling differences, or formatting issues.

### 2.5 Redundant Features
Identify columns that are unlikely to help prediction.

---
## 3. Train/Test Split

| Subsection | Description |
|---|---|
| [3.1 Define X and y](#2.1-Define-X-and-y) | Separate features from the target. |
| [3.2 Train/Test Split](#2.2-Train/Test-Split) | Reserve test data for final evaluation. |

### 3.1 Define X and y
Break down our dataset into **X** (features) and **y** (target variable).

In [ ]:
# Identify Target features:
target_col = "target_col"

# Create X and y dataframes
X = df.drop(columns=target_col)
y = df[target_col]


### 3.2 Train/Test Split
Create the train/test split. Note that from here on we will only be inspecting the training set, to avoid snooping bias.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, y_train, X_test, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 50,
    shuffle = True,
    stratify = y
)

# OPTIONAL: create a training dataframe
train_df = X_train.copy()
train_df["target"] = y_train

- ```test_size``` defines the split, rule of thumb 80/20 train/test split
- ```random_state``` makes split repeatable
- ```shuffle``` shuffle the data before splitting
- ```stratify``` splits the training and test set evenly based on the chosen features or target

---
## 4. Target Analysis

| Subsection | Description |
|---|---|
| [4.1 Classification Target](#41-classification-target) | Inspect class counts and class proportions. |
| [4.2 Regression Target](#4.2-Regression-Target) | Inspect distribution, spread, skewness, and extreme values. |

Note imbalance, transformations, or metric considerations. 



### 4.2 Regression Target
First, look at a histogram of the target variable, along with it's summary statistics.

In [ ]:
plt.hist(y, bins=25)
plt.axvline(y.mean(), color="red", linestyle="--", label="Mean")
plt.axvline(y.median(), color="black", linestyle="--", label="Median")
plt.title("Target Variable (price) Histogram")
plt.xlabel("price")
plt.ylabel("Counts")
plt.legend()
plt.show()

print(y.agg(["mean", "median", "std", "min", "max", "skew"]).T)

A box plot can help identifying outliers.

In [ ]:
sns.boxplot(y)
plt.title("Target Variable (price) Box Plot")

---
## 5. Univariate Analysis

| Subsection | Description |
|---|---|
| [5.1 Numerical Features](#5.1-Numerical-Features) | Review summary statistics, histograms, and box plots. |
| [5.2 Categorical Features](#5.2-Categorical-Features) | Review category counts, proportions, and cardinality. |




### 5.1 Numerical Features

| Subsection | Description |
|---|---|
| [5.1.1 Summary Statistics](#511-summary-statistics) | Check mean, median, standard deviation, range, and skewness. |
| [5.1.2 Histograms](#512-histograms) | Inspect distribution shape and skew. |
| [5.1.3 Box Plots](#513-box-plots) | Identify spread and potential extreme values. |


#### Summary Statistics
First pick out numerical columns and print summary statistics:

In [ ]:
num_cols = ["col_1", "col_2", ...] # Replace with chosen columns

print(y.agg(["mean", "median", "std", "min", "max", "skew"]).T)

for col in num_cols:
    print(col)
    print(X_train[col].agg(["mean", "median", "std", "min", "max", "skew"]).T)
    print()

#### Histograms

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(12, 12),
    squeeze=False
)

axes = axes.flatten()
for idx, col in enumerate(num_cols):
    axes[idx].hist(df[col].dropna(), bins=30)
    axes[idx].axvline(df[col].mean(), color="red", linestyle="--", label=f"Mean: {df[col].mean():.2f}")
    axes[idx].axvline(df[col].median(), color="green", linestyle="--", label=f"Median: {df[col].median():.2f}")
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("Count")
    axes[idx].legend()

plt.suptitle("Numerical Feature Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


**What to look for:**
- skewness
- multimodality
- spread
- unusual gaps
- potential transformations
- mean vs median

Note:
- |skewness| < 0.5 → roughly symmetric, usually no transformation needed
- 0.5 to 1.0 → moderate skew; investigate, but don't transform automatically
- \> 1.0 → strong skew; a transformation is worth considering
- \> 2.0 → very strong skew; transformation is often helpful

#### Box Plots
If we notice any potential outliers, we can box-plot our numerical features. This may be useful even if there are not any obvious outliers.

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(12, 12),
    squeeze=False
)

axes = axes.flatten()

for idx, col in enumerate(num_cols):
    sns.boxplot(y=df[col], ax=axes[idx])
    axes[idx].set_title(f"{col} — Box Plot")
    
plt.suptitle("Outlier Detection via Box Plots", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()



### 5.2 Categorical Features

| Subsection | Description |
|---|---|
| [5.2.1 Basic Statistics](#521-cardinality) | Count unique values. |
| [5.2.2 Frequency Plots](#522-frequency-plots) | Compare category counts and proportions. Identify low-frequency levels. |


#### Basic Statistics
First pick out categorical columns and print summary statistics:

In [ ]:
cat_cols = [
    "col_1",
    "col_2",
    ...
]

X_train.describe(include="str")

#### Frequency Plots
A simple set of bar graphs that visualize categorical features. (Change normalize to False for counts, instead of proportion.)

In [ ]:

fig, axes = plt.subplots(
    2, 4,
    figsize=(16, 12),
    squeeze=False
)
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    counts = np.round(df[col].value_counts(normalize=True),2)
    axes[idx].bar(counts.index.astype(str), counts.values)
    axes[idx].set_title(f"Distribution of {col}")
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("Count")
    for j, v in enumerate(counts.values):
        axes[idx].text(j, v + 0.01, str(v), ha="center", fontweight="bold")

plt.suptitle("Categorical Feature Distributions", fontsize=14, fontweight="bold")
#plt.tight_layout()
plt.show()

**What to look for:**
- which categories are common/rare
- imbalanced features
- unexpected categories
- low-frequency levels (may require grouping)

---
## 6. Feature/Target Relationships

| Subsection | Description |
|---|---|
| [6.1 Numerical vs Classification Target](#61-numerical-vs-classification-target) | Compare numerical feature distributions across classes. |
| [6.2 Categorical vs Classification Target](#62-categorical-vs-classification-target) | Compare positive-class rates across categories. |
| [6.3 Numerical vs Regression Target](#6.3-Numerical-vs-Regression-Target) | Use scatter plots and correlations. |
| [6.4 Categorical vs Regression Target](#6.4-Categorical-vs-Regression-Target) | Compare target distributions across categories. |



### 6.3 Numerical vs Regression Target
**Scatter plot:**

In [ ]:
fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 10),
    squeeze=False
)

axes = axes.flatten()
for idx, col in enumerate(num_cols):
    axes[idx].scatter(X_train[col], y_train)
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("target variable")
    axes[idx].set_title(f"{col} vs target variable")

plt.suptitle("Numerical Features vs Target Variable", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Correlation matrix:** This gives the correlations between features and the target variable, and also includes feature-feature correlations. Take a note of very high feature-feature correlations, as this may either signify redundant variables (which may require dropping) or correlations which may require polynomial features.

In [ ]:
target_var = "target_variable"

numeric_df = df[num_cols].copy()
numeric_df[target_var] = y_train

corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))  # Upper triangle mask
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=1, ax=ax)
ax.set_title("Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

If you want to visualize just the correlations with the target variable, use the following:

In [ ]:
# Correlation with target
target_corr = corr[target_var].drop(target_var).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors)
ax.set_title(f"Correlation with {target_var}")
ax.set_xlabel("Correlation Coefficient")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

### 6.4 Categorical vs Regression Target
look for:
- categories with noticeably different target values
- rare categories that may need grouping
- suspicious or anomalous categories
- categories with large spread/outliers
- features that look potentially predictive
- features that may need encoding or simplification

#### Bar Graphs

In [ ]:
cat_df = df[cat_cols].copy()
cat_df[target_var] = y_train

fig, axes = plt.subplots(
    2, 4,
    figsize=(16, 12),
    squeeze=False
)

axes = axes.flatten()

for idx, col in enumerate(cat_cols): 
    df_temp = cat_df.groupby(col)[target_var].mean()
    axes[idx].bar(df_temp.index, df_temp.values)
    axes[idx].set_title(f"{target_var} by {col}")
    axes[idx].set_ylabel(target_var)
    axes[idx].axhline(y=df[target_var].mean(), color="red", linestyle="--", label="Overall Rate")
    axes[idx].legend()
    
plt.suptitle("Categorical Feature vs Price", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


#### Box Plots
To see if any outliers reside mainly in a single feature vs target variable relation, we can use box plots:

In [ ]:
fig, axes = plt.subplots(
    2, 4,
    figsize=(16, 12),
    squeeze=False
)

axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    sns.boxplot(
        data=cat_df,
        x=col,
        y=target_var,
        ax=axes[idx]
    )

    axes[idx].set_title(f"{target_var} by {col}")
    axes[idx].tick_params(axis="x", rotation=45)

plt.suptitle(
    "Target Distribution by Categorical Feature",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

#### Binary Category Correlations
If we have some categorical features that are *binary*, we can easily convert these into a 0/1 encoding and look at the correlations with the target variable.

In [ ]:
binary_cols = [
    "bin_col_1",
    "bin_col_2",
    ...
]

binary_df = X_train[binary_cols].copy()

# Encode Yes/No as 1/0
# Make sure no/yes have the correct syntax
binary_df[binary_cols] = binary_df[binary_cols].replace({
    "no": 0,
    "yes": 1
})

# Add target
binary_df[target_var] = y_train

# Correlation matrix
corr_matrix = binary_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Upper triangle mask
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=1, ax=ax)
ax.set_title("Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 7. Feature/Feature Relationships

| Subsection | Description |
|---|---|
| [7.1 Correlation Matrix](#71-correlation-matrix) | Identify strong numerical relationships and redundancy. |
| [7.2 Selected Scatter Plots](#72-selected-scatter-plots) | Investigate important relationships in more detail. |


---
## 8. Outliers and Skew

| Subsection | Description |
|---|---|
| [8.1 Potential Outliers](#81-potential-outliers) | Flag unusual observations without automatically deleting them. |
| [8.2 Skewed Features](#82-skewed-features) | Identify variables that may benefit from transformation. |
| [8.3 Proposed Treatment](#83-proposed-treatment) | Record whether values should be retained, transformed, capped, or investigated. |



We can test out potential feature transforms by inspecting how the correlation is effected.

In [ ]:
# First pick out our columns (these can be numeric or categorical)
trans_cols = ["col_1", ...]

X_trans = X_train[trans_cols].copy()

for col in trans_cols:
    X_trans[col] = np.log1p(X_trans[col])
    # Alternative transforms:
    # np.sqrt()
    # np.cbrt()
    # rob_scal = RobustScaler()
    #X_trans = rob_scal.fit_transform(X_train)


X_trans["target_variable"] = y_train

corr = X_trans.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))  # Upper triangle mask
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=1, ax=ax)
ax.set_title("Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 9. Feature Engineering Ideas

| Subsection | Description |
|---|---|
| [9.1 Ratios and Differences](#91-ratios-and-differences) | Identify meaningful derived numerical features. |
| [9.2 Date Features](#92-date-features) | Extract useful components from dates or durations. |
| [9.3 Category Grouping](#93-category-grouping) | Combine rare or logically similar categories. |
| [9.4 Domain Features](#94-domain-features) | Record features suggested by subject knowledge. |



In [ ]:
X_eng = pd.DataFrame()

X_eng["new_col"] = X_train["col_1"] + X_train["col_2"]
X_eng["new_col"] = X_train["col_1"] / X_train["col_2"]

# Include price for correlation matrix
X_eng["target_variable"] = y_train

corr = X_eng.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))  # Upper triangle mask
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=1, ax=ax)
ax.set_title("Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 10. EDA Conclusions

| Subsection | Description |
|---|---|
| [10.1 Key Findings](#101-key-findings) | Summarize the main patterns discovered. |
| [10.2 Features to Drop](#102-features-to-drop) | Record columns that should likely be excluded. |
| [10.3 Missing-Value Strategy](#103-missingvalue-strategy) | Decide how missing data should be handled. |
| [10.4 Encoding and Scaling](#104-encoding-and-scaling) | Record preprocessing requirements. |
| [10.5 Transformations](#105-transformations) | Record transformations worth testing. |
| [10.6 Feature Engineering Plan](#106-feature-engineering-plan) | Record the new features to create. |